# Sinh dữ liệu

Hàm cần xấp xỉ:

$$f(x) = \frac{1}{2\pi\sqrt{\det C}} \exp\left(-\frac{1}{2}(x-\mu)^T C^{-1}(x-\mu)\right)$$

với $\mu = (0, 0)$, $C = \begin{bmatrix}0.3 & -0.5\\-0.5 & 2\end{bmatrix}$, $x_1 \in [-2, 2]$, $x_2 \in [-4, 4]$.

Nhóm sinh 3 tập: train_pool 2000 mẫu, val 200 mẫu, test 100 mẫu. x1, x2 lấy ngẫu nhiên đều trong miền, seed 42.
Dữ liệu chỉ sinh một lần. Nếu trong `data/` đã có file thì notebook không ghi lại mà chỉ kiểm tra xem sinh lại có ra đúng như cũ không.

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import integrate
from scipy.stats import multivariate_normal

# tim thu muc goc BTL/ de chay duoc tu BTL/ hoac tu preprocessing/
ROOT = Path.cwd() if (Path.cwd() / "preprocessing").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from preprocessing.gaussian import MU, COV, gaussian_2d, read_split

DATA_DIR = ROOT / "data"
SEED = 42
SIZES = {"train_pool": 2000, "val": 200, "test": 100}

## 1. Kiểm tra hàm Gaussian

In [2]:
print("det(C) =", np.linalg.det(COV))
print("C^-1 =\n", np.linalg.inv(COV))
print("tri rieng cua C:", np.linalg.eigvalsh(COV))

f00 = gaussian_2d([0, 0])[0]
print("f(0,0) =", f00, "| ly thuyet:", 1 / (2 * np.pi * np.sqrt(0.35)))
assert abs(f00 - 0.269) < 1e-3

# so voi scipy
pts = np.random.default_rng(0).uniform([-2, -4], [2, 4], size=(10000, 2))
max_err = np.max(np.abs(gaussian_2d(pts) - multivariate_normal(MU, COV).pdf(pts)))
print("sai lech lon nhat so voi scipy:", max_err)
assert max_err < 1e-12

det(C) = 0.3499999999999999
C^-1 =
 [[5.71428571 1.42857143]
 [1.42857143 0.85714286]]
tri rieng cua C: [0.16384585 2.13615415]
f(0,0) = 0.26902095463038056 | ly thuyet: 0.26902095463038056
sai lech lon nhat so voi scipy: 1.1102230246251565e-16


In [3]:
# tich phan tren mien rat rong phai ~ 1, tren mien de bai thi nho hon 1 mot chut
f = lambda x2, x1: gaussian_2d([x1, x2])[0]
total, _ = integrate.dblquad(f, -10, 10, -20, 20)
inside, _ = integrate.dblquad(f, -2, 2, -4, 4)
print("tich phan tren mien rong:", round(total, 6))
print("tich phan tren [-2,2]x[-4,4]:", round(inside, 6))

tich phan tren mien rong: 1.0
tich phan tren [-2,2]x[-4,4]: 0.995151


## 2. Sinh dữ liệu

Dùng chung một bộ sinh số ngẫu nhiên, rút lần lượt train_pool, val, test. Mỗi tập rút hết x1 rồi mới rút x2.

In [4]:
rng = np.random.default_rng(SEED)
splits = {}
for name, n in SIZES.items():
    x1 = rng.uniform(-2, 2, n)
    x2 = rng.uniform(-4, 4, n)
    y = gaussian_2d(np.column_stack([x1, x2]))
    splits[name] = pd.DataFrame({"x1": x1, "x2": x2, "y": y})

In [5]:
paths = {name: DATA_DIR / f"{name}.csv" for name in SIZES}
exists = [p.exists() for p in paths.values()]

if all(exists):
    # da co du lieu -> khong ghi de, chi kiem tra sinh lai co giong khong
    for name, p in paths.items():
        assert np.array_equal(read_split(p).to_numpy(), splits[name].to_numpy()), name + " khong khop"
    print("data/ da co san va khop voi seed", SEED, "-> khong ghi de")
elif any(exists):
    raise RuntimeError("data/ chi con mot phan file, kiem tra lai truoc khi sinh")
else:
    DATA_DIR.mkdir(exist_ok=True)
    for name, df in splits.items():
        # ghi du 17 chu so va xuong dong LF de file giong nhau tren moi may
        df.to_csv(paths[name], index=False, float_format="%.17g", lineterminator="\n")
    print("da luu", list(paths))

data/ da co san va khop voi seed 42 -> khong ghi de


## 3. Kiểm tra dữ liệu đã lưu

In [6]:
data = {name: read_split(p) for name, p in paths.items()}
for name, df in data.items():
    X = df[["x1", "x2"]].to_numpy()
    print(name, df.shape,
          "| NaN:", df.isna().sum().sum(),
          "| x1:", df.x1.min().round(3), "..", df.x1.max().round(3),
          "| x2:", df.x2.min().round(3), "..", df.x2.max().round(3),
          "| y max:", df.y.max().round(4))
    assert df.x1.between(-2, 2).all() and df.x2.between(-4, 4).all()
    assert np.allclose(df.y, gaussian_2d(X), rtol=1e-12, atol=0)

train_pool (2000, 3) | NaN: 0 | x1: -1.998 .. 1.998 | x2: -3.995 .. 3.998 | y max: 0.2675
val (200, 3) | NaN: 0 | x1: -1.975 .. 1.986 | x2: -3.976 .. 3.986 | y max: 0.2661
test (100, 3) | NaN: 0 | x1: -1.991 .. 1.985 | x2: -3.987 .. 3.921 | y max: 0.2494


In [7]:
# kiem tra trung diem trong 1 tap va giua cac tap (tranh ro ri du lieu)
pts = {name: set(map(tuple, df[["x1", "x2"]].to_numpy())) for name, df in data.items()}
for name in pts:
    assert len(pts[name]) == len(data[name]), name + " co diem trung"
print("train_pool & val :", len(pts["train_pool"] & pts["val"]))
print("train_pool & test:", len(pts["train_pool"] & pts["test"]))
print("val & test       :", len(pts["val"] & pts["test"]))

train_pool & val : 0
train_pool & test: 0
val & test       : 0


In [8]:
# luu seed, kich thuoc va MD5 cua tung file de sau nay biet file co bi sua khong
md5 = {p.name: hashlib.md5(p.read_bytes().replace(b"\r\n", b"\n")).hexdigest() for p in paths.values()}
meta_path = DATA_DIR / "metadata.json"
if meta_path.exists():
    assert json.loads(meta_path.read_text())["md5"] == md5, "file CSV da bi thay doi"
else:
    meta_path.write_text(json.dumps({"seed": SEED, "sizes": SIZES, "md5": md5}, indent=2))
md5

{'train_pool.csv': 'd5002417dfd0b7b9020167e67289c261',
 'val.csv': '386275d59879c9907ff47402e70fd350',
 'test.csv': '36871f529de6b7182f2c23674de368ee'}